# FakeShield AI - Kaggle training notebook (clean version)

Order: GPU check, project setup, data check, 5 experiments, results table, figures, export.
Prerequisites: *Accelerator = GPU T4*, *Internet = On*, and these inputs added to the notebook:
the `fakeshield-code` dataset (project), the `fakeddit-tsv` dataset (TSV files) and the Fakeddit images notebook.

In [ ]:
import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 1. Project setup

In [ ]:
!rm -rf /kaggle/working/FakeShield-AI
!cp -r /kaggle/input/datasets/chaymaeelbahloul06/fakeshield-code/FakeShield-AI /kaggle/working/
%cd /kaggle/working/FakeShield-AI
!pip install -q captum transformers scikit-learn tqdm
!ls

## 2. Paths and data check

In [ ]:
import os, pandas as pd
TSV = "/kaggle/input/datasets/chaymaeelbahloul06/fakeddit-tsv/multimodal_only_samples/multimodal_train.tsv"
IMG = "/kaggle/input/notebooks/vanshikavmittal/fakeddit-multimodal-fake-news-classification/images"

df = pd.read_csv(TSV, sep="\t", usecols=["id", "2_way_label"])
ids = {f[:-4] for f in os.listdir(IMG) if f.endswith(".jpg")}
m = df[df["id"].astype(str).isin(ids)]
print("rows with image:", len(m), "| expected split 70/15/15 =", int(len(m)*.7), int(len(m)*.15), int(len(m)*.15))
print(m["2_way_label"].value_counts())

## 3. Quick smoke test (about 1 minute)

In [ ]:
COMMON = f"--single_tsv {TSV} --image_dir {IMG} --batch_size 32"
!python -m src.training.train {COMMON} --mode multimodal --fusion concat --limit 500 --epochs 1 --batch_size 16

## 4. The 5 experiments (about 1 hour in total)
Tip: use *Save Version > Save & Run All* so it keeps running in the background.

In [ ]:
COMMON = f"--single_tsv {TSV} --image_dir {IMG} --epochs 3 --batch_size 32"
!python -m src.training.train {COMMON} --mode text
!python -m src.training.train {COMMON} --mode image
!python -m src.training.train {COMMON} --mode multimodal --fusion concat
!python -m src.training.train {COMMON} --mode multimodal --fusion gated
!python -m src.training.train {COMMON} --mode multimodal --fusion attention

## 5. Optional: extra seeds for the multimodal models
Gated, concat and attention are close, so repeating with other seeds shows whether the gap is real.
Uncomment to run (about 2 hours for 2 more seeds).

In [ ]:
# for seed in (1, 2):
#     for fusion in ("concat", "gated", "attention"):
#         !python -m src.training.train {COMMON} --mode multimodal --fusion {fusion} --seed {seed}

## 6. Results table

In [ ]:
import json, glob, pandas as pd
rows = []
for f in sorted(glob.glob("reports/results/*.json")):
    r = json.load(open(f)); t = r["test"]
    rows.append({"run": r["model"], "seed": r.get("seed", 42), "accuracy": t["accuracy"],
                 "precision_macro": t["precision_macro"], "recall_macro": t["recall_macro"],
                 "f1_macro": t["f1_macro"], "roc_auc": t.get("roc_auc"),
                 "ms_per_sample": t["inference_ms_per_sample"], "params_M": r["n_params"] / 1e6})
res = pd.DataFrame(rows).round(4)
res["model"] = res["run"].str.replace(r"_s\d+$", "", regex=True)
display(res.drop(columns="model"))
if res.groupby("model").size().max() > 1:
    display(res.groupby("model")[["accuracy", "f1_macro", "roc_auc"]].agg(["mean", "std"]).round(4))

## 7. Figures (saved in `reports/figures/`)

In [ ]:
import numpy as np, matplotlib.pyplot as plt, os
os.makedirs("reports/figures", exist_ok=True)
base = res[res["seed"] == 42].set_index("model")
order = [m for m in ["text", "image", "multimodal_concat", "multimodal_gated", "multimodal_attention"] if m in base.index]

# bar chart
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(order)); w = 0.38
ax.bar(x - w/2, base.loc[order, "accuracy"], w, label="Accuracy")
ax.bar(x + w/2, base.loc[order, "f1_macro"], w, label="Macro-F1")
ax.set_xticks(x); ax.set_xticklabels(order, rotation=20, ha="right"); ax.set_ylim(0.7, 1.0)
ax.set_title("Test performance by model"); ax.legend(); plt.tight_layout()
plt.savefig("reports/figures/model_comparison.png", dpi=200); plt.show()

# confusion matrices
fig, axes = plt.subplots(1, len(order), figsize=(3.2 * len(order), 3.2))
for ax, name in zip(np.atleast_1d(axes), order):
    cm = np.array(json.load(open(f"reports/results/{name}.json"))["test"]["confusion_matrix"])
    ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max()/2 else "black")
    ax.set_title(name, fontsize=9); ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.tight_layout(); plt.savefig("reports/figures/confusion_matrices.png", dpi=200); plt.show()

In [ ]:
# ROC curves (needs the *_test_preds.npz files written by the updated train.py)
from sklearn.metrics import roc_curve, auc
fig, ax = plt.subplots(figsize=(5, 5)); found = False
for name in order:
    p = f"reports/results/{name}_test_preds.npz"
    if os.path.exists(p):
        d = np.load(p); fpr, tpr, _ = roc_curve(d["y_true"], d["y_prob"][:, 1])
        ax.plot(fpr, tpr, label=f"{name} (AUC={auc(fpr, tpr):.3f})"); found = True
if found:
    ax.plot([0, 1], [0, 1], "k--", lw=0.8); ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
    ax.set_title("ROC curves (test)"); ax.legend(fontsize=8); plt.tight_layout()
    plt.savefig("reports/figures/roc_curves.png", dpi=200); plt.show()
else:
    print("No prediction files found. Re-run the experiments with the updated code to get ROC curves.")

## 8. Export (download from the Output tab)

In [ ]:
!zip -rq results.zip reports/results reports/figures
!ls -lh models results.zip